# Experimental Qwen 1,024-token arithmetic A/B swap-ensemble submission
Uses the previously trained private Qwen2.5-0.5B LoRA adapter. Each competition row is inferred twice: original A/B order and swapped B/A order. Swapped probabilities are mapped back to original A/B/tie labels, then averaged arithmetically. **No training** and **Internet off**.

Exploratory local validation on the reused 1,200-row pilot split: single-pass 1,024-token Qwen **1.320317**, arithmetic swap ensemble **1.190046**. The prior single-pass Qwen leaderboard score was **1.35009**; the best project score remains the length-only **1.06530**. This submission tests whether the local ensemble improvement transfers to hidden Kaggle evaluation.


In [ ]:
from pathlib import Path
import importlib, importlib.metadata, subprocess, sys
from packaging.version import Version
try:
    v=importlib.metadata.version('torchao')
except importlib.metadata.PackageNotFoundError:
    v=None
if v is not None and Version(v)<=Version('0.16.0'):
    print('Removing incompatible optional torchao',v,'offline')
    subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True)
    importlib.invalidate_caches()
import torch
if not torch.cuda.is_available(): raise RuntimeError('GPU required')
root=Path('/kaggle/input')
mounted=sorted(p.name for p in root.iterdir()) if root.is_dir() else []
csv_roots=sorted({p.parent for p in root.rglob('test.csv')})
competition=[p for p in csv_roots if 'llm-classification-finetuning' in str(p).lower()]
if not competition and len(csv_roots)==1: competition=csv_roots
if len(competition)!=1: raise FileNotFoundError('Official test.csv mount not uniquely found: '+repr(mounted))
TEST=competition[0]/'test.csv'
bases=[p.parent for p in root.rglob('config.json') if 'qwen2.5' in str(p).lower() and '0.5b' in str(p).lower()]
if len(bases)!=1: raise FileNotFoundError('Qwen base not uniquely found: '+repr(mounted))
BASE=bases[0]
adapters=[p.parent for p in root.rglob('adapter_model.safetensors') if (p.parent/'adapter_config.json').is_file()]
previous=[p for p in adapters if 'llm-preference-qwen05b-lora-pilot' in str(p)]
if not previous and len(adapters)==1: previous=adapters
if len(previous)!=1: raise FileNotFoundError('Saved Qwen adapter not uniquely found: '+repr(mounted))
ADAPTER=previous[0]
print('test',TEST,'base',BASE,'adapter',ADAPTER)


In [ ]:
# Generated by python scripts/sync_qwen_swap_submit_notebook.py; do not edit this cell by hand.
from pathlib import Path
import sys
source_dir=Path('/kaggle/working/src')
source_dir.mkdir(parents=True,exist_ok=True)
(source_dir/'__init__.py').write_text('',encoding='utf-8')
(source_dir/'baseline.py').write_text("\"\"\"Leakage-controlled, swap-augmented TF-IDF baseline for Kaggle LLM preference prediction.\n\nThis is a classical ML baseline, not an LLM fine-tuning run.\n\"\"\"\nimport argparse\nimport ast\nimport json\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom scipy.sparse import csr_matrix, hstack, vstack\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import log_loss\nfrom sklearn.model_selection import train_test_split\n\nTARGETS = [\"winner_model_a\", \"winner_model_b\", \"winner_tie\"]\nTEXT_COLUMNS = [\"prompt\", \"response_a\", \"response_b\"]\n\n\ndef flatten_messages(value, max_chars=2400):\n    \"\"\"Normalize Kaggle's serialized lists of turns; cap length for a CPU starter.\"\"\"\n    if value is None or (isinstance(value, float) and np.isnan(value)):\n        return \"\"\n    if isinstance(value, str):\n        text = value.strip()\n        if text.startswith(\"[\"):\n            try:\n                value = json.loads(text)\n            except (ValueError, TypeError):\n                try:\n                    value = ast.literal_eval(text)\n                except (ValueError, SyntaxError):\n                    value = text\n        else:\n            value = text\n    if isinstance(value, (list, tuple)):\n        text = \" \".join(\"\" if item is None else str(item) for item in value)\n    else:\n        text = str(value)\n    return text[:max_chars]\n\n\ndef normalized_frame(df):\n    missing = [c for c in TEXT_COLUMNS if c not in df]\n    if missing:\n        raise ValueError(f\"Missing text columns: {missing}\")\n    return pd.DataFrame(\n        {col: [flatten_messages(v) for v in df[col]] for col in TEXT_COLUMNS},\n        index=df.index,\n    )\n\n\ndef get_labels(df):\n    missing = [c for c in TARGETS if c not in df]\n    if missing:\n        raise ValueError(f\"Missing label columns: {missing}\")\n    y = df[TARGETS].to_numpy(dtype=int)\n    if not np.all(y.sum(axis=1) == 1) or not np.all((y == 0) | (y == 1)):\n        raise ValueError(\"Expected exactly one binary winner label per training row\")\n    return y.argmax(axis=1)\n\n\ndef flip_pairs(df):\n    flipped = df.copy()\n    flipped[\"response_a\"], flipped[\"response_b\"] = (\n        df[\"response_b\"].copy(), df[\"response_a\"].copy()\n    )\n    return flipped\n\n\ndef make_vectorizer(df):\n    # Only fit on training-partition texts; do not fit on held-out validation/test.\n    min_df = 2 if len(df) >= 30 else 1\n    vectorizer = TfidfVectorizer(\n        ngram_range=(1, 2), max_features=35000, min_df=min_df,\n        strip_accents=\"unicode\", sublinear_tf=True, dtype=np.float32,\n    )\n    vectorizer.fit(\n        df[\"prompt\"].tolist() + df[\"response_a\"].tolist() +\n        df[\"response_b\"].tolist()\n    )\n    return vectorizer\n\n\ndef pair_features(df, vectorizer):\n    q = vectorizer.transform(df[\"prompt\"])\n    a = vectorizer.transform(df[\"response_a\"])\n    b = vectorizer.transform(df[\"response_b\"])\n    len_a = df[\"response_a\"].str.len().to_numpy(dtype=np.float32)\n    len_b = df[\"response_b\"].str.len().to_numpy(dtype=np.float32)\n    len_q = df[\"prompt\"].str.len().to_numpy(dtype=np.float32)\n    numeric = np.column_stack([\n        np.log1p(len_a) - np.log1p(len_b),\n        (np.log1p(len_a) + np.log1p(len_b)) / 2,\n        np.log1p(len_q),\n    ]) / 10.0\n    return hstack([q, a - b, (a + b) * 0.5, csr_matrix(numeric)],\n                  format=\"csr\", dtype=np.float32)\n\n\ndef fit_baseline(df, y):\n    vectorizer = make_vectorizer(df)\n    x_original = pair_features(df, vectorizer)\n    x_flipped = pair_features(flip_pairs(df), vectorizer)\n    swapped_labels = np.where(y == 0, 1, np.where(y == 1, 0, 2))\n    model = LogisticRegression(C=2.0, max_iter=300, random_state=42)\n    model.fit(vstack([x_original, x_flipped], format=\"csr\"),\n              np.concatenate([y, swapped_labels]))\n    return {\"vectorizer\": vectorizer, \"model\": model, \"targets\": TARGETS}\n\n\ndef predict_prob(bundle, df):\n    features = pair_features(df, bundle[\"vectorizer\"])\n    raw = bundle[\"model\"].predict_proba(features)\n    out = np.zeros((len(df), len(TARGETS)), dtype=np.float64)\n    for col_idx, class_idx in enumerate(bundle[\"model\"].classes_):\n        out[:, int(class_idx)] = raw[:, col_idx]\n    return out / out.sum(axis=1, keepdims=True)\n\n\ndef train(train_csv, out_dir, validation_fraction=0.15):\n    raw = pd.read_csv(train_csv)\n    df = normalized_frame(raw)\n    y = get_labels(raw)\n    if len(np.unique(y)) != 3 or np.min(np.bincount(y, minlength=3)) < 2:\n        raise ValueError(\"Each class must have at least two examples for validation\")\n    x_tr, x_val, y_tr, y_val = train_test_split(\n        df, y, test_size=validation_fraction, random_state=42, stratify=y\n    )\n    validation_bundle = fit_baseline(x_tr, y_tr)\n    val_probs = predict_prob(validation_bundle, x_val)\n    metrics = {\n        \"validation_log_loss\": float(log_loss(y_val, val_probs, labels=[0, 1, 2])),\n        \"train_rows\": int(len(x_tr)),\n        \"validation_rows\": int(len(x_val)),\n        \"full_rows\": int(len(df)),\n        \"seed\": 42,\n        \"note\": \"Random stratified split; not a competition leaderboard result.\",\n    }\n    output = Path(out_dir)\n    output.mkdir(parents=True, exist_ok=True)\n    (output / \"validation_metrics.json\").write_text(\n        json.dumps(metrics, indent=2) + \"\\n\", encoding=\"utf-8\"\n    )\n    # Refit using all labeled data only after holding out validation above.\n    joblib.dump(fit_baseline(df, y), output / \"baseline.joblib\")\n    return metrics\n\n\ndef predict(test_csv, model_path, out_csv):\n    test = pd.read_csv(test_csv)\n    if \"id\" not in test.columns:\n        raise ValueError(\"Test CSV must contain id\")\n    bundle = joblib.load(model_path)  # Load only artifacts you created/trust.\n    probabilities = predict_prob(bundle, normalized_frame(test))\n    result = pd.DataFrame(probabilities, columns=TARGETS)\n    result.insert(0, \"id\", test[\"id\"])\n    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)\n    result.to_csv(out_csv, index=False)\n    return result\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    sub = parser.add_subparsers(dest=\"command\", required=True)\n    fit = sub.add_parser(\"train\")\n    fit.add_argument(\"--train\", default=\"data/train.csv\")\n    fit.add_argument(\"--out\", default=\"artifacts\")\n    infer = sub.add_parser(\"predict\")\n    infer.add_argument(\"--test\", default=\"data/test.csv\")\n    infer.add_argument(\"--model\", default=\"artifacts/baseline.joblib\")\n    infer.add_argument(\"--out\", default=\"submission.csv\")\n    args = parser.parse_args()\n    if args.command == \"train\":\n        print(json.dumps(train(args.train, args.out), indent=2))\n    else:\n        print(f\"Wrote {len(predict(args.test, args.model, args.out))} rows: {args.out}\")\n\n\nif __name__ == \"__main__\":\n    main()\n",encoding='utf-8')
(source_dir/'qwen_swap_ensemble.py').write_text("\"\"\"Inference-only A/B swap ensemble for the existing Qwen2.5-0.5B LoRA adapter.\n\nThe same saved adapter is evaluated twice on each held-out pair:\n1) original response order A/B;\n2) swapped order B/A, with class columns mapped back to the original labels.\n\nNo training and no competition submission happen here. Only aggregate JSON is written.\n\"\"\"\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.special import softmax\nfrom sklearn.metrics import log_loss\nfrom sklearn.model_selection import train_test_split\n\nfrom src.baseline import TARGETS, flatten_messages, get_labels, normalized_frame\n\n\ndef render_pair(row):\n    \"\"\"Exact text template used in the original Qwen pilot.\"\"\"\n    question = flatten_messages(row[\"prompt\"], max_chars=1200)\n    a = flatten_messages(row[\"response_a\"], max_chars=2400)\n    b = flatten_messages(row[\"response_b\"], max_chars=2400)\n    return (\n        \"A human gave two chatbots the same user request.\\n\"\n        f\"User request: {question}\\n\"\n        f\"Response A: {a}\\n\"\n        f\"Response B: {b}\\n\"\n        \"Predict whether the human prefers response A, response B, or a tie.\"\n    )\n\n\ndef exact_validation(raw, seed=42, max_validation_rows=1200):\n    \"\"\"Reproduce the exact two-step validation sampling from the original pilot.\"\"\"\n    frame = normalized_frame(raw)\n    labels = get_labels(raw)\n    _, val_frame, _, val_labels = train_test_split(\n        frame, labels, test_size=0.15, random_state=seed, stratify=labels\n    )\n    if max_validation_rows and len(val_frame) > max_validation_rows:\n        val_frame, _, val_labels, _ = train_test_split(\n            val_frame, val_labels, train_size=max_validation_rows,\n            random_state=seed, stratify=val_labels\n        )\n    return val_frame, np.asarray(val_labels, dtype=np.int64)\n\n\ndef swap_frame(frame):\n    out = frame.copy()\n    out[\"response_a\"], out[\"response_b\"] = (\n        frame[\"response_b\"].to_numpy(copy=True),\n        frame[\"response_a\"].to_numpy(copy=True),\n    )\n    return out\n\n\ndef predict_probs(model, tokenizer, frame, device=\"cuda:0\", max_length=1024, batch_size=2):\n    import torch\n    if len(frame) == 0:\n        raise ValueError(\"Inference frame is empty\")\n    if batch_size <= 0:\n        raise ValueError(\"batch_size must be positive\")\n    model.eval()\n    result = []\n    with torch.inference_mode():\n        for start in range(0, len(frame), batch_size):\n            chunk = frame.iloc[start:start + batch_size]\n            texts = [render_pair(row) for row in chunk.to_dict(\"records\")]\n            encoded = tokenizer(\n                texts,\n                truncation=True,\n                max_length=max_length,\n                padding=True,\n                return_tensors=\"pt\",\n            )\n            encoded = {key: value.to(device) for key, value in encoded.items()}\n            logits = model(**encoded).logits.detach().float().cpu().numpy()\n            if logits.ndim != 2 or logits.shape[1] != 3:\n                raise ValueError(f\"Expected Nx3 logits, got {logits.shape}\")\n            result.append(softmax(logits.astype(np.float64), axis=1))\n    probs = np.vstack(result)\n    probs = np.clip(probs, 1e-12, 1.0)\n    probs /= probs.sum(axis=1, keepdims=True)\n    return probs\n\n\ndef align_swapped(probabilities):\n    \"\"\"Map swapped-order classes B/A/tie back onto original A/B/tie.\"\"\"\n    p = np.asarray(probabilities, dtype=np.float64)\n    if p.ndim != 2 or p.shape[1] != 3:\n        raise ValueError(\"Expected Nx3 probabilities\")\n    return p[:, [1, 0, 2]]\n\n\ndef arithmetic_ensemble(original, swapped_aligned):\n    p = 0.5 * (np.asarray(original) + np.asarray(swapped_aligned))\n    p = np.clip(p, 1e-12, None)\n    return p / p.sum(axis=1, keepdims=True)\n\n\ndef geometric_ensemble(original, swapped_aligned):\n    a = np.clip(np.asarray(original, dtype=np.float64), 1e-12, 1.0)\n    b = np.clip(np.asarray(swapped_aligned, dtype=np.float64), 1e-12, 1.0)\n    p = np.sqrt(a * b)\n    p = np.clip(p, 1e-12, None)\n    return p / p.sum(axis=1, keepdims=True)\n\n\ndef summarize(y, p):\n    y = np.asarray(y, dtype=np.int64)\n    p = np.asarray(p, dtype=np.float64)\n    if p.shape != (len(y), 3):\n        raise ValueError(\"Probability matrix shape mismatch\")\n    pred = p.argmax(axis=1)\n    confidence = p.max(axis=1)\n    bins = np.minimum((confidence * 10).astype(int), 9)\n    ece = 0.0\n    for k in range(10):\n        mask = bins == k\n        if not mask.any():\n            continue\n        acc = np.mean(pred[mask] == y[mask])\n        conf = np.mean(confidence[mask])\n        ece += (mask.sum() / len(y)) * abs(acc - conf)\n    return {\n        \"multiclass_log_loss\": float(log_loss(y, p, labels=[0, 1, 2])),\n        \"accuracy\": float(np.mean(pred == y)),\n        \"mean_max_probability\": float(np.mean(confidence)),\n        \"expected_calibration_error_10_bin\": float(ece),\n        \"predicted_counts\": {\n            TARGETS[i]: int(np.sum(pred == i)) for i in range(3)\n        },\n    }\n\n\ndef disagreement(original, swapped_aligned):\n    a = np.asarray(original, dtype=np.float64)\n    b = np.asarray(swapped_aligned, dtype=np.float64)\n    err = np.abs(a - b)\n    return {\n        \"mean_absolute_probability_disagreement\": float(err.mean()),\n        \"fraction_rows_with_any_class_difference_above_0p2\":\n            float(np.mean(err.max(axis=1) > 0.2)),\n        \"fraction_rows_with_changed_predicted_winner\":\n            float(np.mean(a.argmax(axis=1) != b.argmax(axis=1))),\n    }\n\n\ndef run(args):\n    import torch\n    from peft import PeftModel\n    from safetensors import safe_open\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer\n\n    base_dir = Path(args.base_model)\n    adapter_dir = Path(args.adapter)\n    if not (base_dir / \"config.json\").is_file():\n        raise FileNotFoundError(\"Missing Qwen base config\")\n    if not (adapter_dir / \"adapter_config.json\").is_file():\n        raise FileNotFoundError(\"Missing adapter config\")\n    if not (adapter_dir / \"adapter_model.safetensors\").is_file():\n        raise FileNotFoundError(\"Missing adapter weights\")\n    with safe_open(adapter_dir / \"adapter_model.safetensors\", framework=\"pt\", device=\"cpu\") as handle:\n        head_keys = [key for key in handle.keys() if \"score\" in key and key.endswith(\".weight\")]\n    if not head_keys:\n        raise ValueError(\"Saved adapter does not include classifier score weights\")\n    if not torch.cuda.is_available():\n        raise RuntimeError(\"GPU required for inference-only ensemble audit\")\n\n    raw = pd.read_csv(args.train)\n    x_val, y_val = exact_validation(raw, seed=args.seed, max_validation_rows=args.max_validation_rows)\n\n    tokenizer = AutoTokenizer.from_pretrained(\n        base_dir, local_files_only=True, trust_remote_code=False\n    )\n    if tokenizer.pad_token_id is None:\n        if tokenizer.eos_token is None:\n            raise ValueError(\"Tokenizer lacks pad and EOS\")\n        tokenizer.pad_token = tokenizer.eos_token\n\n    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n    base = AutoModelForSequenceClassification.from_pretrained(\n        base_dir,\n        num_labels=3,\n        torch_dtype=dtype,\n        local_files_only=True,\n        trust_remote_code=False,\n    )\n    base.config.pad_token_id = tokenizer.pad_token_id\n    base.config.use_cache = False\n    model = PeftModel.from_pretrained(\n        base, adapter_dir, is_trainable=False, local_files_only=True\n    )\n    model.to(\"cuda:0\")\n    model.eval()\n\n    original = predict_probs(\n        model, tokenizer, x_val,\n        max_length=args.max_length, batch_size=args.batch_size\n    )\n    swapped_raw = predict_probs(\n        model, tokenizer, swap_frame(x_val),\n        max_length=args.max_length, batch_size=args.batch_size\n    )\n    swapped = align_swapped(swapped_raw)\n    arithmetic = arithmetic_ensemble(original, swapped)\n    geometric = geometric_ensemble(original, swapped)\n\n    result = {\n        \"type\": \"saved_qwen_1024_full_swap_ensemble\",\n        \"no_new_training\": True,\n        \"no_competition_submission\": True,\n        \"validation_rows\": int(len(y_val)),\n        \"context_tokens\": int(args.max_length),\n        \"seed\": int(args.seed),\n        \"original\": summarize(y_val, original),\n        \"swapped_aligned\": summarize(y_val, swapped),\n        \"arithmetic_swap_ensemble\": summarize(y_val, arithmetic),\n        \"geometric_swap_ensemble\": summarize(y_val, geometric),\n        \"original_vs_swapped\": disagreement(original, swapped),\n        \"deltas_vs_original\": {\n            \"arithmetic_log_loss\":\n                summarize(y_val, arithmetic)[\"multiclass_log_loss\"] -\n                summarize(y_val, original)[\"multiclass_log_loss\"],\n            \"geometric_log_loss\":\n                summarize(y_val, geometric)[\"multiclass_log_loss\"] -\n                summarize(y_val, original)[\"multiclass_log_loss\"],\n        },\n        \"limits\": [\n            \"The 1,200-row validation set has already been used for context-length selection; \"\n            \"ensemble selection here is exploratory and not independent confirmation.\",\n            \"Averaging original and swapped predictions enforces order symmetry by construction \"\n            \"but cannot prove the human labels themselves are order-invariant.\",\n            \"The saved adapter was trained on only 2,000 original pairs for one epoch.\",\n            \"Any competition submission based on this ensemble should be labeled experimental.\",\n        ],\n    }\n    best_name = min(\n        (\"original\", \"arithmetic_swap_ensemble\", \"geometric_swap_ensemble\"),\n        key=lambda name: result[name][\"multiclass_log_loss\"],\n    )\n    result[\"lowest_observed_log_loss_method\"] = best_name\n    result[\"lowest_observed_log_loss\"] = result[best_name][\"multiclass_log_loss\"]\n\n    Path(args.output).parent.mkdir(parents=True, exist_ok=True)\n    Path(args.output).write_text(\n        json.dumps(result, indent=2) + \"\\n\", encoding=\"utf-8\"\n    )\n    print(json.dumps({\n        \"original\": result[\"original\"],\n        \"swapped_aligned\": result[\"swapped_aligned\"],\n        \"arithmetic\": result[\"arithmetic_swap_ensemble\"],\n        \"geometric\": result[\"geometric_swap_ensemble\"],\n        \"disagreement\": result[\"original_vs_swapped\"],\n        \"best_method\": result[\"lowest_observed_log_loss_method\"],\n        \"best_log_loss\": result[\"lowest_observed_log_loss\"],\n    }, indent=2))\n    return result\n\n\nif __name__ == \"__main__\":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--train\", required=True)\n    parser.add_argument(\"--base-model\", required=True)\n    parser.add_argument(\"--adapter\", required=True)\n    parser.add_argument(\"--output\", default=\"/kaggle/working/qwen_swap_ensemble.json\")\n    parser.add_argument(\"--seed\", type=int, default=42)\n    parser.add_argument(\"--max-validation-rows\", type=int, default=1200)\n    parser.add_argument(\"--max-length\", type=int, default=1024)\n    parser.add_argument(\"--batch-size\", type=int, default=2)\n    run(parser.parse_args())\n",encoding='utf-8')
(source_dir/'qwen_swap_submit_1024.py').write_text("\"\"\"Authorized experimental Kaggle submission: Qwen 1024-token A/B arithmetic ensemble.\n\nUses the existing saved LoRA adapter. Each test pair is inferred twice, once in\noriginal A/B order and once with responses swapped. Swapped probabilities are\nmapped back to the original label order and averaged arithmetically.\nNo training occurs.\n\"\"\"\nimport argparse\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom src.baseline import TARGETS, normalized_frame\nfrom src.qwen_swap_ensemble import (\n    align_swapped,\n    arithmetic_ensemble,\n    predict_probs,\n    swap_frame,\n)\n\n\ndef run(args):\n    import torch\n    from peft import PeftModel\n    from safetensors import safe_open\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer\n\n    base_dir = Path(args.base_model)\n    adapter_dir = Path(args.adapter)\n    if not (base_dir / \"config.json\").is_file():\n        raise FileNotFoundError(\"Missing Qwen base config\")\n    if not (adapter_dir / \"adapter_config.json\").is_file():\n        raise FileNotFoundError(\"Missing saved adapter config\")\n    if not (adapter_dir / \"adapter_model.safetensors\").is_file():\n        raise FileNotFoundError(\"Missing saved adapter weights\")\n    with safe_open(\n        adapter_dir / \"adapter_model.safetensors\", framework=\"pt\", device=\"cpu\"\n    ) as handle:\n        head_keys = [k for k in handle.keys() if \"score\" in k and k.endswith(\".weight\")]\n    if not head_keys:\n        raise ValueError(\"Saved adapter does not contain classifier score weights\")\n    if not torch.cuda.is_available():\n        raise RuntimeError(\"Kaggle GPU is required for ensemble competition inference\")\n\n    test = pd.read_csv(args.test)\n    if \"id\" not in test.columns:\n        raise ValueError(\"Competition test.csv requires id\")\n    frame = normalized_frame(test)\n\n    tokenizer = AutoTokenizer.from_pretrained(\n        base_dir, local_files_only=True, trust_remote_code=False\n    )\n    if tokenizer.pad_token_id is None:\n        if tokenizer.eos_token is None:\n            raise ValueError(\"Tokenizer lacks pad and EOS\")\n        tokenizer.pad_token = tokenizer.eos_token\n\n    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n    base = AutoModelForSequenceClassification.from_pretrained(\n        base_dir,\n        num_labels=3,\n        torch_dtype=dtype,\n        local_files_only=True,\n        trust_remote_code=False,\n    )\n    base.config.pad_token_id = tokenizer.pad_token_id\n    base.config.use_cache = False\n    model = PeftModel.from_pretrained(\n        base, adapter_dir, is_trainable=False, local_files_only=True\n    )\n    model.to(\"cuda:0\")\n    model.eval()\n\n    original = predict_probs(\n        model, tokenizer, frame,\n        max_length=args.max_length, batch_size=args.batch_size\n    )\n    swapped_raw = predict_probs(\n        model, tokenizer, swap_frame(frame),\n        max_length=args.max_length, batch_size=args.batch_size\n    )\n    swapped_aligned = align_swapped(swapped_raw)\n    ensemble = arithmetic_ensemble(original, swapped_aligned)\n\n    if ensemble.shape != (len(test), 3):\n        raise ValueError(f\"Expected {len(test)}x3 ensemble probabilities, got {ensemble.shape}\")\n    if not np.isfinite(ensemble).all() or np.any(ensemble < 0):\n        raise ValueError(\"Invalid ensemble probabilities\")\n    if not np.allclose(ensemble.sum(axis=1), 1.0, atol=1e-6):\n        raise ValueError(\"Ensemble probabilities are not normalized\")\n\n    submission = pd.DataFrame(ensemble, columns=TARGETS)\n    submission.insert(0, \"id\", test[\"id\"])\n    out = Path(args.output)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    submission.to_csv(out, index=False)\n\n    disagreement = np.abs(original - swapped_aligned)\n    print(f\"Wrote {len(submission)} ensemble rows to {out}\")\n    print(\"Context tokens:\", args.max_length)\n    print(\"Batch size:\", args.batch_size)\n    print(\"Preview mean original-vs-swapped probability disagreement:\", float(disagreement.mean()))\n    print(\"Preview changed-winner fraction:\", float(\n        np.mean(original.argmax(axis=1) != swapped_aligned.argmax(axis=1))\n    ))\n    return len(submission)\n\n\nif __name__ == \"__main__\":\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\"--test\", required=True)\n    p.add_argument(\"--base-model\", required=True)\n    p.add_argument(\"--adapter\", required=True)\n    p.add_argument(\"--output\", default=\"/kaggle/working/submission.csv\")\n    p.add_argument(\"--max-length\", type=int, default=1024)\n    p.add_argument(\"--batch-size\", type=int, default=2)\n    run(p.parse_args())\n",encoding='utf-8')
sys.path.insert(0,'/kaggle/working')
from src.qwen_swap_submit_1024 import run


In [ ]:
from argparse import Namespace
args=Namespace(test=str(TEST),base_model=str(BASE),adapter=str(ADAPTER),output='/kaggle/working/submission.csv',max_length=1024,batch_size=2)
rows=run(args)
print('Submission rows:',rows)
